# Era 2: 2001-2011 

In [ ]:
import sys
# sys.path.append('../src')

In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 

In [ ]:
df_raw = pd.read_excel("../data/A-2001/F2001-01.xls")

In [ ]:
print(df_raw.columns.tolist())
df_raw['Año'].unique()

The dataset contains rows named "total" that summarize data for a specific day. I have to remove these rows and create another dataframe to separate them. 

In [ ]:
mask_total = df_raw["Año"]== "Total"

df_total = df_raw[mask_total]
df = df_raw[~mask_total]

In [ ]:
print(len(df_raw))
print(len(df))
print(len(df_total))

Let's rename the columns: 

In [ ]:
df.isna().sum()

I have to remove columns that aren't being used. 

In [ ]:
cols_to_drop = ['Hora', 'T. máx.', 'Hora.1', 'T. mín.', 'Hora.2', 'HR. mx.', 'Hora.3', 'HR. mn.', 'Hora.4']
df = df.drop(columns= cols_to_drop)

In [ ]:
new_names = [
    "year",
    "month",
    "day",
    "H",
    "rain_mm",
    "temp",
    "humidity",
    "vapor_pressure",
    "dew_point",
]

In [ ]:
print(df.columns)
df.columns = new_names
print(df.columns)

The `"#H"` column needs parsing beacause the time format is corrupted: 

For example 1540.0 should be 15:40. Later on I will unite every time column into a single timestamp. 

In [ ]:
def parse_hour(raw_value):
    chain = str(int(raw_value)).zfill(4)
    hours = chain[:2]
    minutes = chain[2:]
    return f"{hours}:{minutes}"


df["H"] = df["H"].apply(parse_hour)


In [ ]:
df.head()

In [ ]:
df["H"]

In [ ]:
mask_2400 = df["H"] == "24:00"
df.loc[mask_2400, "H"] = "00:00"

In [ ]:
df["hour"] = df["H"].str[:2]
df["minute"] = df["H"].str[3:5] #: is included when slicing
df["second"] = 0


In [ ]:
df["date"]=pd.to_datetime(df[["year","month","day","hour","minute","second"]]) #column names are standard
df["date"].dtype

In [ ]:
new_order = [
    "date",
    "temp",
    "humidity",
    "vapor_pressure",
    "dew_point",
    "rain_mm",
    "H",
    "year",
    "month",
    "day",
    "hour",
    "minute",
    "second"
]

df = df[new_order]

Let's correct adding one day to every midnight value.

In [ ]:
df.loc[mask_2400,"date"] += pd.Timedelta(days=1)

In [ ]:
df["date"].diff().value_counts().head() #computes difference (has to be 10 min)

In [ ]:
df["date"].is_monotonic_increasing #has to say True because time increases

## EDA before generalizing data pipeline 

EDA stands for exploratory data analysis

In [ ]:
print(df["date"].min())
print(df["date"].min().normalize())

We show nights as grey stripes.

In [ ]:
night_start = 21
night_end = 7


days = pd.date_range(df["date"].min().normalize(), df["date"].max().normalize(), freq = "D")


In [ ]:

plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["temp"], color = "red")
for day in days:
    start = day + pd.Timedelta(hours=night_start)
    end= day + pd.Timedelta(days=1, hours = night_end)
    plt.axvspan(start, end, color= "gray", alpha = 0.15)
plt.xticks(rotation=45)
plt.title("September 2001 Temperature - Salamanca")
plt.ylabel("Temp. ºC")
plt.grid()
plt.tight_layout()
plt.savefig("../images/example_temperature.png", dpi = 200, bbox_inches = "tight")

plt.show()


In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["humidity"], color = "cornflowerblue")
plt.xticks(rotation=45)

plt.title("September 2001 Humidity - Salamanca")
plt.ylabel("Humidity %")
plt.grid()
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(df["date"], df["vapor_pressure"], color = "steelblue")
plt.xticks(rotation=45)
plt.title("September 2001 Vapor Pressure - Salamanca")
plt.ylabel("Vapor Pressure")
plt.grid()
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(14, 4))
plt.bar(df["date"], df["rain_mm"], color = "royalblue")
plt.xticks(rotation=45)
plt.title("September 2001 Rain - Salamanca")
plt.ylabel("Rain (mm)")
plt.grid()
plt.tight_layout()

plt.show()